# Fix Missing Text Locus Keys

Somehow I neglected to retain the `text_locus` key for most of the apparatus entries, so I need to restore it. I'm going to do that by making groups of entries and claims. I'll copy the `target` value of the claims into the `text_locus` value.

In [34]:
import json

In [35]:
file = "../converted-json/merged-apparatus-entries-and-claims.json"
with open(file, "r", encoding="utf-8") as f:
    data = json.load(f)

In [36]:
entries = [entry for entry in data if entry.get("type") == "apparatus_entry"] 

In [37]:
def get_apparatus_claims(entry, data=data):
    claims = []
    for claim in data:
        if claim.get("type") == "apparatus_claim" and claim.get("apparatus_entry_id") == entry.get("id"):
            claims.append(claim)
    return claims


def get_text_locus(entry):
    claims = get_apparatus_claims(entry)
    text_locus = claims[0].get("target") if claims else None
    if text_locus:
        return text_locus
    else:
        return None

In [38]:
for entry in entries:
    if not entry.get("text_locus"):
        text_locus = get_text_locus(entry)
        
        if text_locus:
            entry["text_locus"] = text_locus
            print(f"Updated entry {entry['id']} with text locus: {text_locus}")
        else:
            print(f"No text locus found for entry {entry['id']}")

Updated entry urn:uuid:f2e93fa5-f815-4ff4-92ca-adaa365f8f20 with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.1@C.']
Updated entry urn:uuid:37b666e6-9980-4776-81d9-d056936f49c5 with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.1@Nondum']
Updated entry urn:uuid:9de244d4-7413-47c6-a20d-ddc6c86ddefc with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.14@nocturnaque']
Updated entry urn:uuid:bc016a02-5c44-4fc2-8b56-9273dc2e2bfd with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.15@pervia']
Updated entry urn:uuid:059560c1-2d15-4988-8320-0b81080d38a8 with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.15@Fauni']
Updated entry urn:uuid:0380e1d1-ab68-4b6d-855d-89803f846b88 with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.16@prome igitur calamos et si qua recondita servas;']
Updated entry urn:uuid:ac90f88d-aeb8-4389-ba83-b85092f1bfd8 with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.16@']
Updated entry urn:uuid:57e3c173-eb80-46e2-bc35-9370b845300b wit

Okay, that proof of concept worked. Now I'll modify the JSON itself.

In [39]:
for entry in data:
    if entry.get("type") == "apparatus_entry" and not entry.get("text_locus"):
        print(f"Entry {entry['id']} is missing text locus.")
        text_locus = get_text_locus(entry)
        if text_locus:
            entry["text_locus"] = text_locus
            print(f"Updated entry {entry['id']} with text locus: {text_locus}")
        else:
            print(f"No text locus found for entry {entry['id']}")
    if entry.get("type") == "apparatus_entry" and entry.get("target"):
        print(f"Entry {entry['id']} has a target field: {entry['target']}")
        text_locus = get_text_locus(entry)
        if text_locus:
            entry["text_locus"] = text_locus
            entry.pop("target", None)
            print(f"Updated entry {entry['id']} with text locus: {text_locus} and removed target field.")
        else:
            print(f"No text locus found for entry {entry['id']}, target field remains.")


Entry urn:uuid:9de244d4-7413-47c6-a20d-ddc6c86ddefc has a target field: ['urn:cts:latinLit:phi0830.phi001.dll:1.14@nocturnaque']
Updated entry urn:uuid:9de244d4-7413-47c6-a20d-ddc6c86ddefc with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.14@nocturnaque'] and removed target field.
Entry urn:uuid:bc016a02-5c44-4fc2-8b56-9273dc2e2bfd has a target field: ['urn:cts:latinLit:phi0830.phi001.dll:1.15@pervia']
Updated entry urn:uuid:bc016a02-5c44-4fc2-8b56-9273dc2e2bfd with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.15@pervia'] and removed target field.
Entry urn:uuid:059560c1-2d15-4988-8320-0b81080d38a8 has a target field: ['urn:cts:latinLit:phi0830.phi001.dll:1.15@Fauni']
Updated entry urn:uuid:059560c1-2d15-4988-8320-0b81080d38a8 with text locus: ['urn:cts:latinLit:phi0830.phi001.dll:1.15@Fauni'] and removed target field.
Entry urn:uuid:0380e1d1-ab68-4b6d-855d-89803f846b88 has a target field: ['urn:cts:latinLit:phi0830.phi001.dll:1.16']
Updated entry urn:uuid:0380e1d1-ab68-4

In [40]:
new_file = "../converted-json/merged-apparatus-entries-and-claims-updated.json"
with open(new_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)